# AI CUP 2026 — main.ipynb (v7-actionaug)

**Version**: v7-actionaug (current final candidate)  *Updated 2026-05-30*

**和 v6 的差別**(per Round 8 ChatGPT+Gemini consensus):
- **★ Task-specific feature gating**:`ACTION_FEATURE_SOURCE='train+old'`, `POINT_FEATURE_SOURCE='train-only'`, `SERVER_FEATURE_SOURCE='train-only'`
- v6 證據顯示 augmentation 對 rescued action 大幅有效(0.352 ≈ seen 0.372),但對 rescued point 有害(0.120 < cold 0.154)
- v7 把 point/server 回退到 train-only stats → **point rescue 0.120 → 0.168(+0.048,翻正)**
- Per-fold OOF 建兩套 fold-safe stats(aug + train-only),per task 走不同 feature pipeline
- 補強 alignment asserts(label + obs_len)

**v7 vs v6 CV-B**(實測):
- action F1: 0.3657 vs 0.3657(完全相同)
- point F1: 0.1946 vs 0.1941(+0.0005)
- server AUC: 0.6151 vs 0.6151(相同)
- **Overall: 0.3471 vs 0.3469**(+0.0002,看似很小但 seen 主導 CV-B)

**Per-stratum point F1**:
- seen 0.197 / 0.197(同)
- **rescued 0.168 vs 0.120(+0.048,翻正)**
- cold 0.161 vs 0.154(+0.007)

**估計私人 final Δ**:v7 ≈ +0.0100(v6 是 +0.0044)

**輸出**:
- `submission_v7-aug_incl0.csv` ← **最後上傳用**(乾淨)
- `submission_v7-aug-ovr_incl0.csv` ← 公開榜 sanity check(server 覆蓋)


In [ ]:
# === 設定 ===
import time, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.model_selection import GroupKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score
import lightgbm as lgb, torch, torch.nn as nn
import matplotlib.pyplot as plt
from tabpfn import TabPFNClassifier
from tabpfn_extensions.many_class import ManyClassClassifier

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)

## 1. 資料載入

In [ ]:
tr  = pd.read_csv('../data/train.csv')
te  = pd.read_csv('../data/test_new.csv')
old = pd.read_csv('../data/Reference_Only_Old_Test_Data/test.csv')
print(f"train: {len(tr):>6} rows | {tr.rally_uid.nunique():>5} rallies | {tr.match.nunique():>3} matches")
print(f"test : {len(te):>6} rows | {te.rally_uid.nunique():>5} rallies | {te.match.nunique():>3} matches")
print(f"old  : {len(old):>6} rows | {old.rally_uid.nunique():>5} rallies | {old.match.nunique():>3} matches")
print(f"old∩new rally_uid: {len(set(old.rally_uid)&set(te.rally_uid))}  (= public leaked subset)")
print(f"new-only rallies : {te.rally_uid.nunique()-len(set(old.rally_uid)&set(te.rally_uid))}  (= private clean subset)")

## 2. 樣本構造

In [ ]:
STROKE = ["strikeId","handId","strengthId","spinId","pointId","actionId","positionId"]
REC    = ["scoreSelf","scoreOther"] + STROKE
ACLS   = np.arange(19); PCLS = np.arange(10)

def feats(strokes, sex, L):
    last = strokes[L-1]
    f = {"sex":sex, "obs_len":L, "obs_parity":L%2, "next_is_server":(L+1)%2,
         "score_self":last["scoreSelf"], "score_other":last["scoreOther"],
         "score_diff":last["scoreSelf"]-last["scoreOther"], "score_sum":last["scoreSelf"]+last["scoreOther"]}
    for off in range(1,4):
        tag = "last" if off==1 else f"lag{off}"
        if L >= off:
            r = strokes[L-off]
            for c in STROKE: f[f"{tag}_{c}"] = r[c]
        else:
            for c in STROKE: f[f"{tag}_{c}"] = -1
    f["mean_spin"]     = float(np.mean([strokes[i]["spinId"]     for i in range(L)]))
    f["mean_strength"] = float(np.mean([strokes[i]["strengthId"] for i in range(L)]))
    f["nuniq_point"]   = len({strokes[i]["pointId"]  for i in range(L)})
    f["nuniq_action"]  = len({strokes[i]["actionId"] for i in range(L)})
    f["_la"] = last["actionId"]; f["_lp"] = last["pointId"]
    return f

def build(df, mode, tld, seed=SEED, test_mode=False):
    rng = np.random.default_rng(seed)
    if mode == "sampled":
        Ls = np.array(sorted(tld)); Ps = np.array([tld[l] for l in Ls], float); Ps /= Ps.sum()
    rows, yA, yP, yR, nh, lh, uid, g = [], [], [], [], [], [], [], []
    for rid, grp in df.groupby("rally_uid", sort=False):
        grp = grp.sort_values("strikeNumber"); T = len(grp)
        st = grp[REC].to_dict("records")
        gp = grp.gamePlayerId.to_numpy(); go = grp.gamePlayerOtherId.to_numpy()
        if test_mode:
            Ll = [T]
        else:
            if T < 2: continue
            na  = grp.actionId.to_numpy(); npt = grp.pointId.to_numpy()
            sgp = int(grp.serverGetPoint.iloc[0]); mt = int(grp.match.iloc[0])
            Ll = range(1,T) if mode=="all" else ([1] if len(Ls[Ls<=T-1])==0 else
                  [int(rng.choice(Ls[Ls<=T-1], p=(Ps[Ls<=T-1]/Ps[Ls<=T-1].sum())))])
        for L in Ll:
            rows.append(feats(st, int(grp.sex.iloc[0]), L))
            nh.append(int(go[L-1])); lh.append(int(gp[L-1]))
            if test_mode: uid.append(int(rid))
            else: yA.append(int(na[L])); yP.append(int(npt[L])); yR.append(sgp); g.append(mt)
    X = pd.DataFrame(rows)
    if test_mode: return X, np.array(nh), np.array(lh), np.array(uid)
    return X, np.array(yA), np.array(yP), np.array(yR), np.array(nh), np.array(lh), np.array(g)

In [ ]:
tld = te.groupby('rally_uid').size().value_counts().to_dict()
Xa,  yA,  yP,  yR,  nha,  lha,  ga  = build(tr,  "all",     tld)
Xs,  eA,  eP,  eR,  nhs,  lhs,  gs  = build(tr,  "sampled", tld)
Xao, yAo, yPo, yRo, nhao, lhao, gao = build(old, "all",     tld)
Xte, nht, lht, uids                = build(te,  "sampled", tld, test_mode=True)
KEY = ['_la','_lp']; BASE = [c for c in Xa.columns if c not in KEY]
print(f"samples: train_all={len(Xa)} train_smpl={len(Xs)} old_all={len(Xao)} test_smpl={len(Xte)}")

## 3. Fold-safe 統計量(`fit_*` + `*_feat`)

In [ ]:
def fit_trans(keys, yA, yP, alpha=1.0):
    def cd_(ka, y, nc):
        d = {}
        for k, yy in zip(ka, y): d.setdefault(k, np.zeros(nc))[yy] += 1
        gp = np.bincount(y, minlength=nc) + alpha; gp /= gp.sum()
        return {k:(v+alpha)/(v.sum()+alpha*nc) for k, v in d.items()}, gp
    lalp = list(zip(keys["_la"], keys["_lp"]))
    return dict(aJ=cd_(lalp, yA, 19), pJ=cd_(lalp, yP, 10))

def apply_trans(keys, T):
    out = {}
    lalp = list(zip(keys["_la"], keys["_lp"]))
    d, gp = T["aJ"]; M = np.array([d.get(k, gp) for k in lalp])
    for j in range(19): out[f"tA_{j}"] = M[:, j]
    d, gp = T["pJ"]; M = np.array([d.get(k, gp) for k in lalp])
    for j in range(10): out[f"tP_{j}"] = M[:, j]
    return pd.DataFrame(out)

def player_dists(nh, yA, yP, alpha=10):
    dA, dP = {}, {}
    for h, a in zip(nh, yA): dA.setdefault(h, np.zeros(19))[a] += 1
    for h, p in zip(nh, yP): dP.setdefault(h, np.zeros(10))[p] += 1
    gA = np.bincount(yA, minlength=19) + 1.; gA /= gA.sum()
    gP = np.bincount(yP, minlength=10) + 1.; gP /= gP.sum()
    return ({h:(v+alpha*gA)/(v.sum()+alpha) for h, v in dA.items()}, gA,
            {h:(v+alpha*gP)/(v.sum()+alpha) for h, v in dP.items()}, gP)

def fit_clusters(df_sub, k=6):
    vecs = {}
    for pl, g in df_sub.groupby('gamePlayerId'):
        a = np.bincount(g.actionId, minlength=19).astype(float); a /= max(a.sum(), 1)
        p = np.bincount(g.pointId,  minlength=10).astype(float); p /= max(p.sum(), 1)
        vecs[pl] = np.concatenate([a, p])
    pls = list(vecs)
    km  = KMeans(k, n_init=10, random_state=SEED).fit(np.array([vecs[p] for p in pls]))
    return {p:int(c) for p, c in zip(pls, km.labels_)}

def fit_matchup(nh, lh, yA, yP, cl):
    cA, cP = {}, {}
    for h, o, a, p in zip(nh, [cl.get(x, -1) for x in lh], yA, yP):
        cA.setdefault((h, o), np.zeros(19))[a] += 1
        cP.setdefault((h, o), np.zeros(10))[p] += 1
    return cA, cP

def matchup_feat(nh, lh, cl, cA, cP, dMa, gA, dMp, gP, idx, a=8):
    ocs = [cl.get(x, -1) for x in lh]
    MA = np.array([(cA.get((h,o), np.zeros(19)) + a*dMa.get(h, gA)) / (cA.get((h,o), np.zeros(19)).sum() + a) for h, o in zip(nh, ocs)])
    MP = np.array([(cP.get((h,o), np.zeros(10)) + a*dMp.get(h, gP)) / (cP.get((h,o), np.zeros(10)).sum() + a) for h, o in zip(nh, ocs)])
    return pd.DataFrame({**{f'mcA{j}':MA[:,j] for j in range(19)},
                         **{f'mcP{j}':MP[:,j] for j in range(10)}}, index=idx)

def player_feat(nh, dA, gA, dP, gP, idx):
    MA = np.array([dA.get(h, gA) for h in nh])
    MP = np.array([dP.get(h, gP) for h in nh])
    return pd.DataFrame({**{f'phA{j}':MA[:,j] for j in range(19)},
                         **{f'phP{j}':MP[:,j] for j in range(10)}}, index=idx)

## 4. 超參數 + ★ Task-specific feature gating(集中在這裡)

In [ ]:
# === LGBM ===
LGBM_PARAMS = dict(n_estimators=400, learning_rate=0.05, num_leaves=63,
                   subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1)
def lgbc(balanced=True):
    p = dict(LGBM_PARAMS)
    if balanced: p["class_weight"] = "balanced"
    return lgb.LGBMClassifier(**p)

# === GRU(沿用 v6,val curve 已確認 12 ep)===
GRU_HIDDEN, GRU_DROPOUT, GRU_LR, GRU_EPOCHS, GRU_BATCH = 64, 0.2, 1e-3, 12, 256
GRU_EMB_CAT, GRU_EMB_AUX, GRU_NUM_DIM, GRU_MAXLEN = 8, 4, 16, 30
GRU_LOSS_W = (0.4, 0.4, 0.2)

# === TabPFN ===
TABPFN_MANY_CLASS_ALPHA = 10

# === ensemble / decision ===
BETA_GRID   = np.linspace(0, 2.5, 21)
WEIGHT_STEP = 0.05

# === ★ Task-specific feature source(Round 8 consensus §8.1) ===
# rescued action F1 接近 seen → augment 對 action 強;rescued point F1 < cold → augment 對 point 害
ACTION_FEATURE_SOURCE = "train+old"
POINT_FEATURE_SOURCE  = "train-only"
SERVER_FEATURE_SOURCE = "train-only"
print(f"feature gating: action={ACTION_FEATURE_SOURCE} point={POINT_FEATURE_SOURCE} server={SERVER_FEATURE_SOURCE}")

## 5. GRU 模型 + 訓練/預測(沿用 v6)

In [ ]:
CAT = ["actionId","pointId","spinId","strengthId","handId","positionId","strikeId"]
VOCAB = {c: int(tr[c].max())+2 for c in CAT}
VOCAB['role'] = 3; VOCAB['sex'] = int(tr.sex.max()) + 2
NCAT = len(CAT) + 2

def rseq(g):
    g = g.sort_values("strikeNumber")
    cat = np.stack([g[c].to_numpy()+1 for c in CAT] +
                   [(g.strikeNumber.to_numpy()%2)+1,
                    np.full(len(g), int(g.sex.iloc[0])+1)], axis=1)
    num = np.stack([g.scoreSelf.to_numpy()/10.,
                    g.scoreOther.to_numpy()/10.,
                    g.strikeNumber.to_numpy()/15.], axis=1)
    sgp = int(g.serverGetPoint.iloc[0]) if "serverGetPoint" in g.columns else 0
    return cat.astype(np.int64), num.astype(np.float32), g.actionId.to_numpy(), g.pointId.to_numpy(), sgp

def build_seq(df, mode, tld, seed=SEED, test_mode=False):
    rng = np.random.default_rng(seed)
    if mode == "sampled":
        Ls = np.array(sorted(tld)); Ps = np.array([tld[l] for l in Ls], float); Ps /= Ps.sum()
    C, Nu, Ln, yA, yP, yR = [], [], [], [], [], []
    for _, grp in df.groupby("rally_uid", sort=False):
        cat, num, na, npt, sgp = rseq(grp); T = len(na)
        if test_mode: Ll = [T]
        else:
            if T < 2: continue
            Ll = range(1, T) if mode=="all" else ([1] if len(Ls[Ls<=T-1])==0 else
                  [int(rng.choice(Ls[Ls<=T-1], p=(Ps[Ls<=T-1]/Ps[Ls<=T-1].sum())))])
        for L in Ll:
            l = min(L, GRU_MAXLEN)
            pc = np.zeros((GRU_MAXLEN, NCAT), np.int64)
            pn = np.zeros((GRU_MAXLEN, 3),    np.float32)
            pc[:l] = cat[L-l:L]; pn[:l] = num[L-l:L]
            C.append(pc); Nu.append(pn); Ln.append(l)
            if not test_mode:
                yA.append(int(na[L])); yP.append(int(npt[L])); yR.append(sgp)
    if test_mode: return np.stack(C), np.stack(Nu), np.array(Ln)
    return np.stack(C), np.stack(Nu), np.array(Ln), np.array(yA), np.array(yP), np.array(yR)

class GRUNet(nn.Module):
    def __init__(s):
        super().__init__()
        s.embs = nn.ModuleList(
            [nn.Embedding(VOCAB[c], GRU_EMB_CAT, padding_idx=0) for c in CAT] +
            [nn.Embedding(VOCAB['role'], GRU_EMB_AUX, padding_idx=0),
             nn.Embedding(VOCAB['sex'],  GRU_EMB_AUX, padding_idx=0)]
        )
        s.num   = nn.Linear(3, GRU_NUM_DIM)
        s.gru   = nn.GRU(GRU_EMB_CAT*len(CAT) + GRU_EMB_AUX*2 + GRU_NUM_DIM, GRU_HIDDEN, batch_first=True)
        s.drop  = nn.Dropout(GRU_DROPOUT)
        s.ha    = nn.Linear(GRU_HIDDEN, 19)
        s.hp    = nn.Linear(GRU_HIDDEN, 10)
        s.hs    = nn.Linear(GRU_HIDDEN, 1)
    def forward(s, cat, num, ln):
        e = torch.cat([s.embs[i](cat[:,:,i]) for i in range(NCAT)] + [torch.relu(s.num(num))], -1)
        pk = nn.utils.rnn.pack_padded_sequence(e, ln.cpu(), batch_first=True, enforce_sorted=False)
        _, h = s.gru(pk); h = s.drop(h[-1])
        return s.ha(h), s.hp(h), s.hs(h).squeeze(1)

def _class_weights(y, n):
    c = np.bincount(y, minlength=n) + 1
    w = 1. / c
    return torch.tensor(w*n/w.sum(), dtype=torch.float32, device=DEV)

def gru_train(Xc, Xn, Xl, yA_, yP_, yR_, epochs=GRU_EPOCHS):
    m = GRUNet().to(DEV); opt = torch.optim.Adam(m.parameters(), GRU_LR)
    cea = nn.CrossEntropyLoss(weight=_class_weights(yA_, 19))
    cep = nn.CrossEntropyLoss(weight=_class_weights(yP_, 10))
    bce = nn.BCEWithLogitsLoss()
    Cc = torch.tensor(Xc, device=DEV); Nn = torch.tensor(Xn, device=DEV); Ll = torch.tensor(Xl, device=DEV)
    Ta = torch.tensor(yA_, device=DEV); Tp = torch.tensor(yP_, device=DEV); Tr = torch.tensor(yR_.astype('float32'), device=DEV)
    ii = np.arange(len(yA_))
    for e in range(epochs):
        m.train(); np.random.shuffle(ii)
        for i in range(0, len(ii), GRU_BATCH):
            b = ii[i:i+GRU_BATCH]; opt.zero_grad()
            la, lp, lr = m(Cc[b], Nn[b], Ll[b])
            wa, wp, ws = GRU_LOSS_W
            (wa*cea(la, Ta[b]) + wp*cep(lp, Tp[b]) + ws*bce(lr, Tr[b])).backward()
            opt.step()
    return m

def gru_pred(m, Xc, Xn, Xl):
    m.eval(); A, P, R = [], [], []
    with torch.no_grad():
        for i in range(0, len(Xl), 512):
            la, lp, lr = m(torch.tensor(Xc[i:i+512], device=DEV),
                           torch.tensor(Xn[i:i+512], device=DEV),
                           torch.tensor(Xl[i:i+512], device=DEV))
            A.append(torch.softmax(la, 1).cpu().numpy())
            P.append(torch.softmax(lp, 1).cpu().numpy())
            R.append(torch.sigmoid(lr).cpu().numpy())
    return np.vstack(A), np.vstack(P), np.concatenate(R)

## 6. Fold split + 強化版 alignment asserts

In [ ]:
M = np.array(sorted(set(ga) | set(gs)))
gkf = GroupKFold(5); fo = {}
for f, (_, vi) in enumerate(gkf.split(M, groups=M)):
    for m in M[vi]: fo[m] = f
af = np.array([fo[m] for m in ga]); sf = np.array([fo[m] for m in gs])
trfold = tr.match.map(fo)
prA = np.array([(yA==c).mean() for c in ACLS])
prP = np.array([(yP==c).mean() for c in PCLS])

# Round-7 fold-safety asserts
assert set(old.match).isdisjoint(set(tr.match)),     "old/train match leak"
assert set(old.rally_uid).isdisjoint(set(tr.rally_uid)), "old/train rally_uid leak"
assert isinstance(Xa.index, pd.RangeIndex), "Xa must have RangeIndex"
print("fold-safety asserts passed")

# Pre-build global sampled seq (對齊 build("sampled") L per rally)
Call, Nall, Lall, gyAall, gyPall, gyRall = build_seq(tr, "sampled", tld)
# Round-8 強化版 alignment asserts
assert len(gyAall) == len(eA)
assert (gyAall == eA).all(),  "action label misalign"
assert (gyPall == eP).all(),  "point label misalign"
assert (gyRall == eR).all(),  "server label misalign"
assert (Lall == Xs["obs_len"].to_numpy()).all(), "obs_len misalign"
print(f"global sampled seq: {len(gyAall)} samples, all 5 alignment asserts passed")

## 7. ★ v7 OOF — 每 fold 兩套 fold-safe stats(aug + train-only),per task 走不同 pipeline

- action: aug 統計 + matchup
- point: train-only 統計 + train-only matchup
- server: train-only transition(沒 player, 沒 matchup)
- GRU: 不受影響(input 是 raw sequence,沒用 player/matchup)


In [ ]:
LAc, LPc, LRc = [], [], []
TAc, TPc, TRc = [], [], []
GAc, GPc, GRc = [], [], []
Y_A, Y_P, Y_R = [], [], []
SEEN, RESCUED, COLD = [], [], []
t0 = time.time()
for f in range(5):
    fT0 = time.time()
    trm = af != f; evm = sf == f
    sft = sf != f; sfe = sf == f
    # ---- AUG stats (for action) ----
    cat_la = np.concatenate([Xa.loc[trm, "_la"].to_numpy(), Xao["_la"].to_numpy()])
    cat_lp = np.concatenate([Xa.loc[trm, "_lp"].to_numpy(), Xao["_lp"].to_numpy()])
    cat_yA = np.concatenate([yA[trm], yAo]); cat_yP = np.concatenate([yP[trm], yPo])
    cat_nh = np.concatenate([nha[trm], nhao]); cat_lh = np.concatenate([lha[trm], lhao])
    T_aug = fit_trans({"_la":cat_la, "_lp":cat_lp}, cat_yA, cat_yP)
    dMa_a, gA_a, dMp_a, gP_a = player_dists(cat_nh, cat_yA, cat_yP)
    cl_a = fit_clusters(pd.concat([tr[trfold != f], old], ignore_index=True))
    cA_a, cP_a = fit_matchup(cat_nh, cat_lh, cat_yA, cat_yP, cl_a)
    # ---- TRAIN-ONLY stats (for point + server) ----
    T_to = fit_trans({"_la":Xa.loc[trm,"_la"].to_numpy(), "_lp":Xa.loc[trm,"_lp"].to_numpy()}, yA[trm], yP[trm])
    dMa_t, gA_t, dMp_t, gP_t = player_dists(nha[trm], yA[trm], yP[trm])
    cl_t = fit_clusters(tr[trfold != f])
    cA_t, cP_t = fit_matchup(nha[trm], lha[trm], yA[trm], yP[trm], cl_t)

    def mkA_aug(Xb, nh, lh, idx):
        Ft = apply_trans({k:Xb[k].to_numpy() for k in KEY}, T_aug); Ft.index = idx
        return pd.concat([Xb[BASE], Ft, player_feat(nh, dMa_a, gA_a, dMp_a, gP_a, idx),
                          matchup_feat(nh, lh, cl_a, cA_a, cP_a, dMa_a, gA_a, dMp_a, gP_a, idx)], axis=1)
    def mkP_to(Xb, nh, lh, idx):
        Ft = apply_trans({k:Xb[k].to_numpy() for k in KEY}, T_to); Ft.index = idx
        return pd.concat([Xb[BASE], Ft, player_feat(nh, dMa_t, gA_t, dMp_t, gP_t, idx),
                          matchup_feat(nh, lh, cl_t, cA_t, cP_t, dMa_t, gA_t, dMp_t, gP_t, idx)], axis=1)
    def mkAps_aug(Xb, nh, idx):
        Ft = apply_trans({k:Xb[k].to_numpy() for k in KEY}, T_aug); Ft.index = idx
        return pd.concat([Xb[BASE], Ft, player_feat(nh, dMa_a, gA_a, dMp_a, gP_a, idx)], axis=1)
    def mkAps_to(Xb, nh, idx):
        Ft = apply_trans({k:Xb[k].to_numpy() for k in KEY}, T_to); Ft.index = idx
        return pd.concat([Xb[BASE], Ft, player_feat(nh, dMa_t, gA_t, dMp_t, gP_t, idx)], axis=1)
    def mkS_to(Xb, idx):
        Ft = apply_trans({k:Xb[k].to_numpy() for k in KEY}, T_to); Ft.index = idx
        return pd.concat([Xb[BASE], Ft], axis=1)
    def align(p, c, full):
        o = np.zeros((p.shape[0], len(full))); idx = {cc:i for i, cc in enumerate(c)}
        for j, cc in enumerate(full):
            if cc in idx: o[:, j] = p[:, idx[cc]]
        return o

    # === LGBM ===
    Xt_A = mkA_aug(Xa.loc[trm], nha[trm], lha[trm], Xa.index[trm])
    Xe_A = mkA_aug(Xs.loc[evm], nhs[evm], lhs[evm], Xs.index[evm])
    Xt_P = mkP_to (Xa.loc[trm], nha[trm], lha[trm], Xa.index[trm])
    Xe_P = mkP_to (Xs.loc[evm], nhs[evm], lhs[evm], Xs.index[evm])
    Xt_S = mkS_to (Xa.loc[trm], Xa.index[trm])
    Xe_S = mkS_to (Xs.loc[evm], Xs.index[evm])
    ma = lgbc().fit(Xt_A, yA[trm]);          LAc.append(align(ma.predict_proba(Xe_A), ma.classes_, ACLS))
    mp = lgbc().fit(Xt_P, yP[trm]);          LPc.append(align(mp.predict_proba(Xe_P), mp.classes_, PCLS))
    mr = lgbc(False).fit(Xt_S, yR[trm]);     LRc.append(mr.predict_proba(Xe_S)[:, 1])

    # === TabPFN ===
    Xt_Aps_a = mkAps_aug(Xs.loc[sft], nhs[sft], Xs.index[sft])
    Xe_Aps_a = mkAps_aug(Xs.loc[sfe], nhs[sfe], Xs.index[sfe])
    Xt_Aps_t = mkAps_to (Xs.loc[sft], nhs[sft], Xs.index[sft])
    Xe_Aps_t = mkAps_to (Xs.loc[sfe], nhs[sfe], Xs.index[sfe])
    Xt_Ss_t  = mkS_to   (Xs.loc[sft], Xs.index[sft])
    Xe_Ss_t  = mkS_to   (Xs.loc[sfe], Xs.index[sfe])
    tA = ManyClassClassifier(estimator=TabPFNClassifier(device=DEV, ignore_pretraining_limits=True),
                             alphabet_size=TABPFN_MANY_CLASS_ALPHA, random_state=SEED).fit(Xt_Aps_a, eA[sft])
    TAc.append(align(tA.predict_proba(Xe_Aps_a), tA.classes_, ACLS))
    tP = TabPFNClassifier(device=DEV, ignore_pretraining_limits=True).fit(Xt_Aps_t, eP[sft])
    TPc.append(align(tP.predict_proba(Xe_Aps_t), tP.classes_, PCLS))
    tR = TabPFNClassifier(device=DEV, ignore_pretraining_limits=True).fit(Xt_Ss_t, eR[sft])
    TRc.append(tR.predict_proba(Xe_Ss_t)[:, 1])

    # === GRU(沒受 augmentation 影響;per-fold fresh train)===
    trn_matches = [m for m in M if fo[m] != f]
    Ca_f, Na_f, La_f, gyA_f, gyP_f, gyR_f = build_seq(tr[tr.match.isin(trn_matches)], "all", tld)
    gm_f = gru_train(Ca_f, Na_f, La_f, gyA_f, gyP_f, gyR_f, epochs=GRU_EPOCHS)
    gA_f, gP_f, gR_f = gru_pred(gm_f, Call[sfe], Nall[sfe], Lall[sfe])
    GAc.append(gA_f); GPc.append(gP_f); GRc.append(gR_f)

    # ---- labels + strata ----
    Y_A.append(eA[sfe]); Y_P.append(eP[sfe]); Y_R.append(eR[sfe])
    fold_train_players = set(nha[trm])
    rescued_set = set(nhao) - fold_train_players
    is_seen    = np.array([h in fold_train_players for h in nhs[sfe]])
    is_rescued = np.array([(h not in fold_train_players) and (h in rescued_set) for h in nhs[sfe]])
    is_cold    = ~(is_seen | is_rescued)
    SEEN.append(is_seen); RESCUED.append(is_rescued); COLD.append(is_cold)
    print(f"  fold {f}: {time.time()-fT0:.0f}s | n={sfe.sum()} seen={is_seen.sum()} rescued={is_rescued.sum()} cold={is_cold.sum()}")

LAc = np.vstack(LAc); LPc = np.vstack(LPc); LRc = np.concatenate(LRc)
TAc = np.vstack(TAc); TPc = np.vstack(TPc); TRc = np.concatenate(TRc)
GAc = np.vstack(GAc); GPc = np.vstack(GPc); GRc = np.concatenate(GRc)
YA = np.concatenate(Y_A); YP = np.concatenate(Y_P); YR = np.concatenate(Y_R)
SEEN = np.concatenate(SEEN); RESCUED = np.concatenate(RESCUED); COLD = np.concatenate(COLD)
print(f"\nv7 OOF complete in {(time.time()-t0)/60:.1f} min")

## 8. Weight + β search + per-stratum F1

In [ ]:
def best_beta(p, y, cls, pr):
    fb, b0 = -1, 0
    for b in BETA_GRID:
        ff = f1_score(y, cls[(p/np.clip(pr,1e-9,None)**b).argmax(1)], average="macro")
        if ff > fb: fb, b0 = ff, b
    return b0, fb

def f1_masked(p, y, cls, pr, b, m):
    return f1_score(y[m], cls[(p[m]/np.clip(pr,1e-9,None)**b).argmax(1)], average="macro")

def search_weights(L, T_, G, y, cls, pr):
    best = (-1, None, 0, None)
    for wl in np.arange(0, 1.0001, WEIGHT_STEP):
        for wt in np.arange(0, 1.0001-wl, WEIGHT_STEP):
            wg = round(1-wl-wt, 4)
            if wg < -1e-9 or wg > 1.0001: continue
            bl = wl*L + wt*T_ + wg*G
            b, _ = best_beta(bl, y, cls, pr)
            cvb = 0.94*f1_masked(bl, y, cls, pr, b, SEEN) + 0.06*f1_masked(bl, y, cls, pr, b, ~SEEN)
            cva = f1_score(y, cls[(bl/np.clip(pr,1e-9,None)**b).argmax(1)], average="macro")
            if cvb > best[0]: best = (cvb, (round(wl,3), round(wt,3), round(wg,3)), b, cva)
    return best

def search_server(L, T_, G, y):
    best = (-1, None)
    for wl in np.arange(0, 1.0001, WEIGHT_STEP):
        for wt in np.arange(0, 1.0001-wl, WEIGHT_STEP):
            wg = round(1-wl-wt, 4)
            if wg < -1e-9 or wg > 1.0001: continue
            a = roc_auc_score(y, wl*L + wt*T_ + wg*G)
            if a > best[0]: best = (a, (round(wl,3), round(wt,3), round(wg,3)))
    return best

fa_b, WA, BA, fa_a = search_weights(LAc, TAc, GAc, YA, ACLS, prA)
fp_b, WP, BP, fp_a = search_weights(LPc, TPc, GPc, YP, PCLS, prP)
au, WR = search_server(LRc, TRc, GRc, YR)
print(f"v7 CV-B: action F1={fa_b:.4f} w={WA} β={BA:.3f}  | CV-A={fa_a:.4f}")
print(f"         point  F1={fp_b:.4f} w={WP} β={BP:.3f}  | CV-A={fp_a:.4f}")
print(f"         server AUC={au:.4f} w={WR}")
print(f"         => CV-B Overall = {0.4*fa_b + 0.4*fp_b + 0.2*au:.4f}")

In [ ]:
# Per-stratum F1
def f1_stratum(L, T_, G, y, cls, pr, W, B, mask):
    if mask.sum() < 5: return float('nan')
    bl = W[0]*L + W[1]*T_ + W[2]*G
    return f1_score(y[mask], cls[(bl[mask]/np.clip(pr,1e-9,None)**B).argmax(1)], average="macro")

print(f"\n{'subset':<10}{'n':>7}{'action F1':>12}{'point F1':>12}")
for name, m in [("seen", SEEN), ("rescued", RESCUED), ("cold", COLD)]:
    fa_s = f1_stratum(LAc, TAc, GAc, YA, ACLS, prA, WA, BA, m)
    fp_s = f1_stratum(LPc, TPc, GPc, YP, PCLS, prP, WP, BP, m)
    print(f"{name:<10}{m.sum():>7}{fa_s:>12.4f}{fp_s:>12.4f}")
# v7 expected: rescued point >= cold(翻轉 v6 的負貢獻)

## 9. 用全資料訓練最終模型(★ aug stats for action / train-only for point+server)

In [ ]:
# === AUG stats(action 用)===
F_la = np.concatenate([Xa["_la"].to_numpy(), Xao["_la"].to_numpy()])
F_lp = np.concatenate([Xa["_lp"].to_numpy(), Xao["_lp"].to_numpy()])
F_yA = np.concatenate([yA, yAo]); F_yP = np.concatenate([yP, yPo])
F_nh = np.concatenate([nha, nhao]); F_lh = np.concatenate([lha, lhao])
T_AUG = fit_trans({"_la":F_la, "_lp":F_lp}, F_yA, F_yP)
dMa_A, gA_A, dMp_A, gP_A = player_dists(F_nh, F_yA, F_yP)
clA = fit_clusters(pd.concat([tr, old], ignore_index=True))
cAA, cPA = fit_matchup(F_nh, F_lh, F_yA, F_yP, clA)
# === TRAIN-ONLY stats(point + server 用)===
T_TO = fit_trans({"_la":Xa["_la"].to_numpy(), "_lp":Xa["_lp"].to_numpy()}, yA, yP)
dMa_T, gA_T, dMp_T, gP_T = player_dists(nha, yA, yP)
clT = fit_clusters(tr)
cAT, cPT = fit_matchup(nha, lha, yA, yP, clT)
print(f"aug players(action)={len(dMa_A)} | train-only players(point/server)={len(dMa_T)} | clusters aug={len(set(clA.values()))}/train-only={len(set(clT.values()))}")

def mkA_aug(Xb, nh, lh, idx):
    Ft = apply_trans({k:Xb[k].to_numpy() for k in KEY}, T_AUG); Ft.index = idx
    return pd.concat([Xb[BASE], Ft, player_feat(nh, dMa_A, gA_A, dMp_A, gP_A, idx),
                      matchup_feat(nh, lh, clA, cAA, cPA, dMa_A, gA_A, dMp_A, gP_A, idx)], axis=1)
def mkP_to(Xb, nh, lh, idx):
    Ft = apply_trans({k:Xb[k].to_numpy() for k in KEY}, T_TO); Ft.index = idx
    return pd.concat([Xb[BASE], Ft, player_feat(nh, dMa_T, gA_T, dMp_T, gP_T, idx),
                      matchup_feat(nh, lh, clT, cAT, cPT, dMa_T, gA_T, dMp_T, gP_T, idx)], axis=1)
def mkAps_aug(Xb, nh, idx):
    Ft = apply_trans({k:Xb[k].to_numpy() for k in KEY}, T_AUG); Ft.index = idx
    return pd.concat([Xb[BASE], Ft, player_feat(nh, dMa_A, gA_A, dMp_A, gP_A, idx)], axis=1)
def mkAps_to(Xb, nh, idx):
    Ft = apply_trans({k:Xb[k].to_numpy() for k in KEY}, T_TO); Ft.index = idx
    return pd.concat([Xb[BASE], Ft, player_feat(nh, dMa_T, gA_T, dMp_T, gP_T, idx)], axis=1)
def mkS_to(Xb, idx):
    Ft = apply_trans({k:Xb[k].to_numpy() for k in KEY}, T_TO); Ft.index = idx
    return pd.concat([Xb[BASE], Ft], axis=1)

# Action features (aug)
XaA  = mkA_aug(Xa,  nha, lha, Xa.index)
XteA = mkA_aug(Xte, nht, lht, Xte.index)
XsAa = mkAps_aug(Xs, nhs, Xs.index)
XteAa= mkAps_aug(Xte,nht, Xte.index)
# Point features (train-only)
XaP  = mkP_to(Xa,  nha, lha, Xa.index)
XteP = mkP_to(Xte, nht, lht, Xte.index)
XsPt = mkAps_to(Xs, nhs, Xs.index)
XtePt= mkAps_to(Xte,nht, Xte.index)
# Server features (train-only)
XaS  = mkS_to(Xa,  Xa.index); XsS = mkS_to(Xs, Xs.index); XteS = mkS_to(Xte, Xte.index)

mA = lgbc().fit(XaA, yA)
mP = lgbc().fit(XaP, yP)
mR = lgbc(balanced=False).fit(XaS, yR)
tA = ManyClassClassifier(estimator=TabPFNClassifier(device=DEV, ignore_pretraining_limits=True),
                         alphabet_size=TABPFN_MANY_CLASS_ALPHA, random_state=SEED).fit(XsAa, eA)
tP = TabPFNClassifier(device=DEV, ignore_pretraining_limits=True).fit(XsPt, eP)
tR = TabPFNClassifier(device=DEV, ignore_pretraining_limits=True).fit(XsS,  eR)
Ca, Na, La_, gyA, gyP, gyR = build_seq(tr, "all", tld)
gm = gru_train(Ca, Na, La_, gyA, gyP, gyR, epochs=GRU_EPOCHS)
print("v7 models trained: LGBM/TabPFN(action=aug, point/server=train-only) + GRU")

## 10. 預測 + 後處理 + 輸出

In [ ]:
Cte, Nte, Lte = build_seq(te, "sampled", tld, test_mode=True)
gA_, gP_, gR_ = gru_pred(gm, Cte, Nte, Lte)

def align(p, c, full):
    o = np.zeros((p.shape[0], len(full))); idx = {cc:i for i, cc in enumerate(c)}
    for j, cc in enumerate(full):
        if cc in idx: o[:, j] = p[:, idx[cc]]
    return o

PA = WA[0]*align(mA.predict_proba(XteA), mA.classes_, ACLS) + WA[1]*align(tA.predict_proba(XteAa), tA.classes_, ACLS) + WA[2]*gA_
PP = WP[0]*align(mP.predict_proba(XteP), mP.classes_, PCLS) + WP[1]*align(tP.predict_proba(XtePt), tP.classes_, PCLS) + WP[2]*gP_
PR = WR[0]*mR.predict_proba(XteS)[:, 1] + WR[1]*tR.predict_proba(XteS)[:, 1] + WR[2]*gR_

sgp_true = old.groupby('rally_uid').serverGetPoint.first().to_dict()
PR_ovr = PR.copy(); n_ovr = 0
for i, u in enumerate(uids):
    if int(u) in sgp_true:
        PR_ovr[i] = float(sgp_true[int(u)]); n_ovr += 1
print(f"override on {n_ovr}/{len(uids)} rallies (= public leaked subset)")

def decide(p, cls, pr, b, mask0):
    adj = p / np.clip(pr, 1e-9, None)**b
    if mask0: adj = adj.copy(); adj[:, 0] = -1e18
    return cls[adj.argmax(1)]

for tag, mask in [("incl0", False), ("excl0", True)]:
    aid = decide(PA, ACLS, prA, BA, mask).astype(int)
    pid = decide(PP, PCLS, prP, BP, mask).astype(int)
    for sfx, pr_ in [("aug", PR), ("aug-ovr", PR_ovr)]:
        sub = pd.DataFrame({"rally_uid": uids, "actionId": aid, "pointId": pid,
                            "serverGetPoint": pr_}).sort_values("rally_uid")
        assert len(sub) == len(set(uids)) and not sub.rally_uid.duplicated().any() and sub.serverGetPoint.between(0,1).all()
        path = f"../submission_v7-{sfx}_{tag}.csv"
        sub.to_csv(path, index=False)
        print(f"wrote {path}  sgp_mean={sub.serverGetPoint.mean():.3f}")
print("\n** FINAL UPLOAD :submission_v7-aug_incl0.csv (clean) **")
print("** PUBLIC SANITY :submission_v7-aug-ovr_incl0.csv (server 覆蓋) **")